In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
from datasets import load_dataset, Audio
#https://huggingface.co/datasets/GianDiego/latam-spanish-speech-orpheus-tts-24khz
dataset = load_dataset("GianDiego/latam-spanish-speech-orpheus-tts-24khz", split="train")
dataset

Dataset({
    features: ['audio', 'text', 'file_id', 'nationality', 'gender', 'speaker_id'],
    num_rows: 24437
})

In [4]:
len(dataset)

24437

In [5]:
print(dataset)

Dataset({
    features: ['audio', 'text', 'file_id', 'nationality', 'gender', 'speaker_id'],
    num_rows: 24437
})


In [6]:
dataset = dataset.filter(
    lambda nationality, gender: nationality == 'ar' and gender == 'f',
    input_columns=['nationality', 'gender']
)

In [7]:
columns_to_remove = ['file_id', 'nationality', 'gender', 'speaker_id']
dataset = dataset.remove_columns(columns_to_remove)
print(dataset)

Dataset({
    features: ['audio', 'text'],
    num_rows: 3921
})


Solo se usan las voces argentinas femeninas

In [8]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [9]:
from transformers import SpeechT5Processor

checkpoint = "microsoft/speecht5_tts"
processor = SpeechT5Processor.from_pretrained(checkpoint)


In [10]:
tokenizer = processor.tokenizer

In [11]:
dataset[2:5]

e:\IA\5_Bimestre\TTFB\TTS\tts\lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
C:\Users\juanc\AppData\Local\Programs\Python\Python310\lib\inspect.py:869: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):


{'audio': [{'path': 'arf_07973_01243309438.wav',
   'array': array([0., 0., 0., ..., 0., 0., 0.]),
   'sampling_rate': 16000},
  {'path': 'arf_01208_01272386743.wav',
   'array': array([0., 0., 0., ..., 0., 0., 0.]),
   'sampling_rate': 16000},
  {'path': 'arf_06136_01580696558.wav',
   'array': array([0., 0., 0., ..., 0., 0., 0.]),
   'sampling_rate': 16000}],
 'text': ['¿Me podés mandar fotos de la pileta?',
  'Hola Cristina que bueno que pueda hablar con vos.',
  'Tranquilo va a estar todo bien.']}

Let's normalize the dataset, create a column called "normalized_text"

In [12]:
def extract_all_chars(batch):
    all_text = " ".join(batch["text"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}


vocabs = dataset.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=dataset.column_names,
)

dataset_vocab = set(vocabs["vocab"][0])
tokenizer_vocab = {k for k, _ in tokenizer.get_vocab().items()}

Map:   0%|          | 0/3921 [00:00<?, ? examples/s]

In [13]:
dataset_vocab - tokenizer_vocab

{' ', '3', '¡', '¿', 'Á', 'É', 'Ú', 'á', 'í', 'ñ', 'ó', 'ú', 'ü'}

In [14]:
import re

def normalize_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation (except apostrophes)
    text = re.sub(r'[^\w\s\']', '', text)

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text

# Define a function to add the normalized_text column
def add_normalized_text(example):
    example['normalized_text'] = normalize_text(example['text'])
    return example

# Apply the function to the dataset
dataset = dataset.map(add_normalized_text)

# Print the first few examples to verify
print(dataset[2:5])

{'audio': [{'path': 'arf_07973_01243309438.wav', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 16000}, {'path': 'arf_01208_01272386743.wav', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 16000}, {'path': 'arf_06136_01580696558.wav', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 16000}], 'text': ['¿Me podés mandar fotos de la pileta?', 'Hola Cristina que bueno que pueda hablar con vos.', 'Tranquilo va a estar todo bien.'], 'normalized_text': ['me podés mandar fotos de la pileta', 'hola cristina que bueno que pueda hablar con vos', 'tranquilo va a estar todo bien']}


In [15]:
def extract_all_chars(batch):
    all_text = " ".join(batch["normalized_text"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}


vocabs = dataset.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=dataset.column_names,
)

dataset_vocab = set(vocabs["vocab"][0])
tokenizer_vocab = {k for k, _ in tokenizer.get_vocab().items()}

Map:   0%|          | 0/3921 [00:00<?, ? examples/s]

In [16]:
dataset_vocab - tokenizer_vocab

{' ', '3', 'á', 'í', 'ñ', 'ó', 'ú', 'ü'}

In [17]:
replacements = [
    ("á", "a"),   # Spanish a with accent
    ("í", "i"),   # Spanish i with accent
    ("ñ", "ny"),  # Spanish ñ
    ("ó", "o"),   # Spanish o with accent
    ("ú", "u"),   # Spanish u with accent
]

def cleanup_text(inputs):
    # Ensure we operate on 'normalized_text'
    if 'normalized_text' in inputs:
        text = inputs["normalized_text"]
        for src, dst in replacements:
            text = text.replace(src, dst)
        inputs["normalized_text"] = text
    return inputs

dataset = dataset.map(cleanup_text)

In [18]:
import os
import torch

from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

spk_model_name = "speechbrain/spkrec-xvect-voxceleb"
device = "cuda" if torch.cuda.is_available() else "cpu"

speaker_model = EncoderClassifier.from_hparams(
    source=spk_model_name,
    run_opts={"device": device},
    savedir=r"E:\IA\5_Bimestre\TTFB\spkrec-xvect-voxceleb",
    local_strategy=LocalStrategy.COPY,
)

def create_speaker_embedding(waveform):
    waveform = torch.tensor(waveform, dtype=torch.float32, device=device)

    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)

    with torch.no_grad():
        speaker_embeddings = speaker_model.encode_batch(waveform)
        speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
        speaker_embeddings = speaker_embeddings.squeeze().cpu().numpy()

    return speaker_embeddings

In [ ]:
# import os
# import torch
# from speechbrain.inference.classifiers import EncoderClassifier

# spk_model_name = "speechbrain/spkrec-xvect-voxceleb"

# device = "cuda" if torch.cuda.is_available() else "cpu"
# speaker_model = EncoderClassifier.from_hparams(
#     source=spk_model_name,
#     run_opts={"device": device},
#     savedir=os.path.join("/content", "spkrec-xvect-voxceleb"),
# )

# def create_speaker_embedding(waveform):
#     waveform = torch.tensor(waveform, dtype=torch.float32, device=device)

#     if waveform.ndim == 1:
#         waveform = waveform.unsqueeze(0)

#     with torch.no_grad():
#         speaker_embeddings = speaker_model.encode_batch(waveform)
#         speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
#         speaker_embeddings = speaker_embeddings.squeeze().cpu().numpy()

#     return speaker_embeddings

In [19]:
def prepare_dataset(example):
    audio = example["audio"]

    example = processor(
        text=example["normalized_text"],
        audio_target=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_attention_mask=False,
    )

    # strip off the batch dimension
    example["labels"] = example["labels"][0]

    # use SpeechBrain to obtain x-vector
    example["speaker_embeddings"] = create_speaker_embedding(audio["array"])

    return example

In [20]:
processed_example = prepare_dataset(dataset[0])
list(processed_example.keys())

['input_ids', 'labels', 'speaker_embeddings']

In [21]:
processed_example["speaker_embeddings"].shape

(512,)

In [22]:
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

Map:   0%|          | 0/3921 [00:00<?, ? examples/s]

In [23]:
def is_not_too_long(input_ids):
    input_length = len(input_ids)
    return input_length < 200

dataset = dataset.filter(is_not_too_long, input_columns=["input_ids"])
len(dataset)

Filter:   0%|          | 0/3921 [00:00<?, ? examples/s]

3921

In [24]:
dataset = dataset.train_test_split(test_size=0.1)

In [25]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        label_features = [{"input_values": feature["labels"]} for feature in features]
        speaker_features = [feature["speaker_embeddings"] for feature in features]

        # collate the inputs and targets into a batch
        batch = processor.pad(
            input_ids=input_ids, labels=label_features, return_tensors="pt"
        )

        # replace padding with -100 to ignore loss correctly
        batch["labels"] = batch["labels"].masked_fill(
            batch.decoder_attention_mask.unsqueeze(-1).ne(1), -100
        )

        # not used during fine-tuning
        del batch["decoder_attention_mask"]

        # round down target lengths to multiple of reduction factor
        if model.config.reduction_factor > 1:
            target_lengths = torch.tensor(
                [len(feature["input_values"]) for feature in label_features]
            )
            target_lengths = target_lengths.new(
                [
                    length - length % model.config.reduction_factor
                    for length in target_lengths
                ]
            )
            max_length = max(target_lengths)
            batch["labels"] = batch["labels"][:, :max_length]

        # also add in the speaker embeddings
        batch["speaker_embeddings"] = torch.tensor(speaker_features)

        return batch

In [26]:
data_collator = TTSDataCollatorWithPadding(processor=processor)

In [27]:
from transformers import SpeechT5ForTextToSpeech

model = SpeechT5ForTextToSpeech.from_pretrained(checkpoint)

In [28]:
from functools import partial

# disable cache during training since it's incompatible with gradient checkpointing
model.config.use_cache = False

# set language and task for generation and re-enable cache
model.generate = partial(model.generate, use_cache=True)

In [29]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.2.2+cu121
12.1
True


In [30]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="speecht5_finetuned_tts_rioplatense",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=500,
    gradient_checkpointing=False,
    fp16=True,
    evaluation_strategy="steps",   # <- antes evaluation_strategy
    per_device_eval_batch_size=2,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    greater_is_better=False,
    label_names=["labels"],
    push_to_hub=True,
)

In [31]:
model.gradient_checkpointing_disable()
print(model.is_gradient_checkpointing)

False


In [33]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    tokenizer=processor,
)

In [34]:
trainer.train()

  0%|          | 0/500 [00:00<?, ?it/s]

{'loss': 1.2027, 'grad_norm': 4.093923091888428, 'learning_rate': 2.4e-05, 'epoch': 0.23}
{'loss': 0.8499, 'grad_norm': 4.532238483428955, 'learning_rate': 4.9e-05, 'epoch': 0.45}
{'loss': 0.7225, 'grad_norm': 5.538041591644287, 'learning_rate': 7.3e-05, 'epoch': 0.68}
{'loss': 0.6666, 'grad_norm': 3.7032248973846436, 'learning_rate': 9.8e-05, 'epoch': 0.91}


  0%|          | 0/197 [00:00<?, ?it/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1876}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warning will be raised to an exception in v4.41.


{'eval_loss': 0.5440118908882141, 'eval_runtime': 70.5898, 'eval_samples_per_second': 5.567, 'eval_steps_per_second': 2.791, 'epoch': 0.91}
{'loss': 0.6203, 'grad_norm': 14.010570526123047, 'learning_rate': 9.425e-05, 'epoch': 1.13}
{'loss': 0.5938, 'grad_norm': 5.603659152984619, 'learning_rate': 8.800000000000001e-05, 'epoch': 1.36}
{'loss': 0.5826, 'grad_norm': 8.698128700256348, 'learning_rate': 8.175000000000001e-05, 'epoch': 1.59}
{'loss': 0.553, 'grad_norm': 3.4974303245544434, 'learning_rate': 7.55e-05, 'epoch': 1.81}


...

{'eval_loss': 0.46291959285736084, 'eval_runtime': 70.129, 'eval_samples_per_second': 5.604, 'eval_steps_per_second': 2.809, 'epoch': 3.63}
{'loss': 0.5001, 'grad_norm': 3.201169967651367, 'learning_rate': 1.925e-05, 'epoch': 3.85}
{'loss': 0.495, 'grad_norm': 3.799713134765625, 'learning_rate': 1.3000000000000001e-05, 'epoch': 4.08}
{'loss': 0.4933, 'grad_norm': 3.0170998573303223, 'learning_rate': 6.750000000000001e-06, 'epoch': 4.31}
{'loss': 0.4865, 'grad_norm': 4.128970146179199, 'learning_rate': 5.000000000000001e-07, 'epoch': 4.54}


  0%|          | 0/197 [00:00<?, ?it/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1876}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warning will be raised to an exception in v4.41.


{'eval_loss': 0.45313963294029236, 'eval_runtime': 70.4241, 'eval_samples_per_second': 5.58, 'eval_steps_per_second': 2.797, 'epoch': 4.54}
{'train_runtime': 7916.6969, 'train_samples_per_second': 2.021, 'train_steps_per_second': 0.063, 'train_loss': 0.5969779014587402, 'epoch': 4.54}


TrainOutput(global_step=500, training_loss=0.5969779014587402, metrics={'train_runtime': 7916.6969, 'train_samples_per_second': 2.021, 'train_steps_per_second': 0.063, 'train_loss': 0.5969779014587402, 'epoch': 4.54})

In [35]:
trainer.push_to_hub()

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1876}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warning will be raised to an exception in v4.41.


CommitInfo(commit_url='https://huggingface.co/jFettnpn/speecht5_finetuned_tts_rioplatense/commit/9df811f7eae818626aea30d4510c394b6a382341', commit_message='End of training', commit_description='', oid='9df811f7eae818626aea30d4510c394b6a382341', pr_url=None, pr_revision=None, pr_num=None)

# Inference

In [36]:
model = SpeechT5ForTextToSpeech.from_pretrained(
    "jFettnpn/speecht5_finetuned_tts_rioplatense"
)

config.json: 0.00B [00:00, ?B/s]

e:\IA\5_Bimestre\TTFB\TTS\tts\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juanc\.cache\huggingface\hub\models--jFettnpn--speecht5_finetuned_tts_rioplatense. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/578M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

In [37]:
example = dataset["test"][304]
speaker_embeddings = torch.tensor(example["speaker_embeddings"]).unsqueeze(0)

In [38]:
text = "hola me llamo agus y estudio inteligencia artificial"

In [39]:
number_words = {
    0: "cero", 1: "uno", 2: "dos", 3: "tres", 4: "cuatro", 5: "cinco", 6: "seis", 7: "siete", 8: "ocho", 9: "nueve",
    10: "diez", 11: "once", 12: "doce", 13: "trece", 14: "catorce", 15: "quince", 16: "dieciséis", 17: "diecisiete",
    18: "dieciocho", 19: "diecinueve", 20: "veinte", 30: "treinta", 40: "cuarenta", 50: "cincuenta", 60: "sesenta", 70: "setenta",
    80: "ochenta", 90: "noventa", 100: "cien", 1000: "mil"
}

def number_to_words(number):
    if number < 20:
        return number_words[number]
    elif number < 100:
        tens, unit = divmod(number, 10)
        return number_words[tens * 10] + (" y " + number_words[unit] if unit else "")
    elif number < 1000:
        hundreds, remainder = divmod(number, 100)
        return (number_words[hundreds] + "cientos" if hundreds > 1 else "cien") + (" " + number_to_words(remainder) if remainder else "")
    elif number < 1000000:
        thousands, remainder = divmod(number, 1000)
        return (number_to_words(thousands) + " mil" if thousands > 1 else "mil") + (" " + number_to_words(remainder) if remainder else "")
    elif number < 1000000000:
        millions, remainder = divmod(number, 1000000)
        return number_to_words(millions) + " millones" + (" " + number_to_words(remainder) if remainder else "")
    elif number < 1000000000000:
        billions, remainder = divmod(number, 1000000000)
        return number_to_words(billions) + " mil millones" + (" " + number_to_words(remainder) if remainder else "")
    else:
        return str(number)

def replace_numbers_with_words(text):

    def replace(match):
        number = int(match.group())
        return number_to_words(number)

    # Find the numbers and change with words.
    result = re.sub(r'\b\d+\b', replace, text)

    return result

In [40]:
# Function to clean up text using the replacement pairs
def cleanup_text(text):
    for src, dst in replacements:
        text = text.replace(src, dst)
    return text

In [46]:
text = "hola me llamo agus, tengo 31 años y estudio inteligencia artificial"

In [47]:
converted_text = replace_numbers_with_words(text)
cleaned_text = cleanup_text(converted_text)
final_text = normalize_text(cleaned_text)
final_text

'hola me llamo agus tengo treinta y uno anyos y estudio inteligencia artificial'

In [48]:
inputs = processor(text=final_text, return_tensors="pt")

In [49]:
from transformers import SpeechT5HifiGan

vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

In [50]:
from IPython.display import Audio
import soundfile as sf

Audio(speech.numpy(), rate=16000)
# Save the audio to a file (e.g., 'output.wav')
sf.write('output.wav', speech.numpy(), 16000)